<a href="https://colab.research.google.com/github/nonohuang0819/kaggle/blob/main/task3/task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# from google.colab import drive
# drive.mount("/content/drive")

import pandas as pd
import sklearn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.svm import SVC
import category_encoders as ce


In [2]:
filepath = "../Train/task3/introml_2024_task32_train.csv"
df = pd.read_csv(filepath)

In [3]:
# 前處理
cols = ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'class'] # 0/1變數, class 分類

for c in cols:
    df[c] = df[c].apply(lambda x: x if x=='?' else int(x[-1]))

df.replace("?", np.nan, inplace=True)

/var/folders/z8/tkcn56l93cb258d6cngvzl1h0000gn/T/ipykernel_69049/4238128312.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace("?", np.nan, inplace=True)


In [4]:
print(df.columns)

f8 = df.loc[:,'f8'] == np.nan
df[f8]

# df[df.loc[:,'f8'].isnull().any(axis=1)]
df[df.isnull().any(axis=1)]
df.isnull().sum()
df.dropna()

Index(['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10',
       'f11', 'f12', 'f13', 'f14', 'f15', 'class'],
      dtype='object')


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,f14,f15,class
0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.5812,2.2748,10.7618,0.1328,0.707,0.7827,1.0644,0.2197,0
1,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.8923,1.1706,2.0248,0.4691,0.5685,1.6567,0.7632,0.5765,0
2,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.1836,0.5778,2.9611,0.7307,0.9065,0.6243,0.7995,7.8733,0
3,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.2957,0.7494,3.4072,0.4306,0.2528,0.4312,0.7853,4.5719,0
4,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.9519,1.3822,4.0772,1.3046,0.5435,1.6834,0.9049,1.9258,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4791,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.6482,0.8203,4.0695,0.2277,1.0551,1.0765,0.4434,2.2287,5
4793,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.3654,1.0973,0.259,0.1595,0.0893,0.187,0.3522,0.5147,5
4795,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0265,1.2058,1.5087,0.1448,0.4024,2.435,0.7447,4.9996,5
4796,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.2645,0.2464,0.1314,0.4813,0.5992,0.1151,0.2646,2.3852,5


In [21]:
from sklearn.impute import KNNImputer
col1 = ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7']
col2 = ['f8', 'f9', 'f10','f11', 'f12', 'f13', 'f14', 'f15']
imputer = KNNImputer(n_neighbors=5)
df_imputed_col2 = imputer.fit_transform(df[col2])
df_imputed_col2

df_imputed_col1 = imputer.fit_transform(df[col1])
df_imputed_col1.round()
df_imputed1 = pd.DataFrame(df_imputed_col1.round(), columns=col1)

# 眾數 .779
df_imputed_col1 = df[col1]
for class_ in range(6):
    for c in col1:
        most_frequent_value = df_imputed_col1[c].mode()[0]
        df_imputed_col1[c].fillna(most_frequent_value, inplace=True)
df_imputed1 = df_imputed_col1


df_imputed2 = pd.DataFrame(df_imputed_col2, columns=col2)

df2=pd.concat([df_imputed1, df_imputed2, df.iloc[:,-1]], axis=1)
df2.dropna()

/var/folders/z8/tkcn56l93cb258d6cngvzl1h0000gn/T/ipykernel_69049/7857304.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_imputed_col1[c].fillna(most_frequent_value, inplace=True)
/var/folders/z8/tkcn56l93cb258d6cngvzl1h0000gn/T/ipykernel_69049/7857304.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_imputed_col1[c].fillna(most_f

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,f14,f15,class
0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.5812,2.2748,10.7618,0.1328,0.7070,0.7827,1.0644,0.2197,0
1,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.8923,1.1706,2.0248,0.4691,0.5685,1.6567,0.7632,0.5765,0
2,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.1836,0.5778,2.9611,0.7307,0.9065,0.6243,0.7995,7.8733,0
3,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.2957,0.7494,3.4072,0.4306,0.2528,0.4312,0.7853,4.5719,0
4,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.9519,1.3822,4.0772,1.3046,0.5435,1.6834,0.9049,1.9258,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4795,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0265,1.2058,1.5087,0.1448,0.4024,2.4350,0.7447,4.9996,5
4796,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.2645,0.2464,0.1314,0.4813,0.5992,0.1151,0.2646,2.3852,5
4797,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.4825,1.2393,3.3957,0.1721,0.3529,2.3861,0.1033,6.1273,5
4798,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.3278,0.8246,4.8168,0.7692,0.0987,1.8784,0.0400,4.1618,5


In [41]:
# 全部 drop 掉
df_dropna = df2.dropna()
X = df_dropna.iloc[:, :-1]
y = df_dropna.iloc[:, -1]

from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(score_func=f_classif, k=15)
X_selected = selector.fit_transform(X, y)
print(selector.get_support(indices=True))

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)





model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# 5. 模型预测
y_pred = model.predict(X_test)

# 6. 性能评估
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 15]
Accuracy: 0.7864583333333334
Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.80      0.86       182
           1       0.74      0.85      0.79       164
           2       0.63      0.70      0.66       142
           3       0.73      0.75      0.74       169
           4       0.86      0.82      0.84       147
           5       0.84      0.80      0.82       156

    accuracy                           0.79       960
   macro avg       0.79      0.79      0.79       960
weighted avg       0.80      0.79      0.79       960



In [ ]:
from xgboost import XGBClassifier



X = df_dropna.iloc[:, :-1]
y = df_dropna.iloc[:, -1] 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb_model = XGBClassifier(scale_pos_weight=1)  # 根據需要調整權重
xgb_model.fit(X_train, y_train)

# 5. 模型预测
y_pred = xgb_model.predict(X_test)

# 6. 性能评估
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

# 2 都很爛 注意權重

/Users/liuzihong/Desktop/大三上/機器學習概論/Code/kaggle/.venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [23:47:58] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)


Accuracy: 0.7739583333333333
Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.80      0.85       182
           1       0.72      0.81      0.76       164
           2       0.66      0.69      0.67       142
           3       0.69      0.73      0.71       169
           4       0.85      0.83      0.84       147
           5       0.85      0.78      0.81       156

    accuracy                           0.77       960
   macro avg       0.78      0.77      0.77       960
weighted avg       0.78      0.77      0.78       960



In [9]:
from sklearn.svm import SVC  # 支持向量機分類器
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# 假設 df 是原始數據，並且 'class' 是目標變數
bool_features = ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7']  # 布林特徵
float_features = ['f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f14', 'f15']  # 浮點特徵
target_column = 'class'  # 目標變數

# 取出特徵和目標變數
X_bool = df2[bool_features]
X_float = df2[float_features]
y = df[target_column]

# 分別標準化浮點數特徵
scaler = StandardScaler()
X_float_scaled = scaler.fit_transform(X_float)

# 合併布林特徵和標準化後的浮點數特徵
X_combined = np.hstack([X_bool, X_float_scaled])  # 合併數據

# 分割訓練集和測試集
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

# 建立 SVM 模型（這裡選擇了 RBF 核心，其他選擇還有線性、Polynomial 等）
svm_model = SVC(kernel='rbf', random_state=42)

# 訓練模型
svm_model.fit(X_train, y_train)

# 預測測試集
y_pred = svm_model.predict(X_test)

# 評估模型
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# 顯示分類報告
print(classification_report(y_test, y_pred))


Accuracy: 0.7083
              precision    recall  f1-score   support

           0       0.90      0.71      0.80       182
           1       0.63      0.77      0.69       164
           2       0.56      0.67      0.61       142
           3       0.65      0.63      0.64       169
           4       0.86      0.74      0.80       147
           5       0.72      0.72      0.72       156

    accuracy                           0.71       960
   macro avg       0.72      0.71      0.71       960
weighted avg       0.73      0.71      0.71       960



In [32]:
ansfile = "../Train/task3/introml_2024_task32_test_NO_answers_shuffled.csv"
ans = pd.read_csv(ansfile)
cols = ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7']
for c in cols:
    ans[c] = ans[c].apply(lambda x: x if x=='?' else int(x[-1]))

ans.replace("?", np.nan, inplace=True)
ans


from sklearn.impute import KNNImputer
col1 = ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7']
col2 = ['f8', 'f9', 'f10','f11', 'f12', 'f13', 'f14', 'f15']
imputer = KNNImputer(n_neighbors=5)
df_imputed_col2 = imputer.fit_transform(ans[col2])
df_imputed_col2

df_imputed_col1 = imputer.fit_transform(ans[col1])
df_imputed_col1.round()

df_imputed1 = pd.DataFrame(df_imputed_col1.round(), columns=col1)
df_imputed2 = pd.DataFrame(df_imputed_col2, columns=col2)

# df2=pd.concat([df_imputed1, df_imputed2, df.iloc[:,-1]], axis=1)
# df2.dropna()

# 5. 模型预测
ans_pred = model.predict(pd.concat([df_imputed1, df_imputed2], axis=1))

trans = np.vectorize(lambda x : "C"+str(x))
ans_pred = trans(ans_pred)
submit = pd.DataFrame({
    "id" : range(0, len(ans_pred)),
    "class" : ans_pred
})

submit.to_csv('../Submit/task3/submission2.csv', index=False)


/var/folders/z8/tkcn56l93cb258d6cngvzl1h0000gn/T/ipykernel_69049/4000049686.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ans.replace("?", np.nan, inplace=True)
/Users/liuzihong/Desktop/大三上/機器學習概論/Code/kaggle/.venv/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


ValueError: X has 16 features, but RandomForestClassifier is expecting 15 features as input.

In [233]:
# Submition : 3 per day
input() # 避免誤點到
!kaggle competitions submit -c introml-nccu-2024-task-3 -f ../Submit/task3/submission.csv -m "Message"

100%|██████████████████████████████████████| 8.30k/8.30k [00:00<00:00, 9.55kB/s]
Successfully submitted to introml@nccu_2024_task3